# Assignment Code: DS-AG-031
# Generative AI - Text Generation and Machine Translation | Assignment

**Q1.** What is Generative AI and what are its primary use cases across industries?
- **Ans:** Generative Artificial Intelligence (Generative AI) is a category of AI systems that can create new and original content such as text, images, audio, video, and code-by learning patterns from large datasets using advanced machine learning models.
- **Primary use cases Across Industries:**
  - Healthcare: Clinical documentation, drug discovery, virtual assistants
  - Business & Marketing: Content generation, customer support, personalization
  - Software Development: Code generation, debugging, documentation
  - Education: Personalized learning, content creation, tutoring
  - Media & Entertainment: Script writing, image/video generation
  - Finance: Financial reporting, fraud detection, risk analysis
  - E-commerce: Product descriptions, recommendations, virtual try-ons

**Q2.** Explain the role of probabilistic modeling in generative models. How do these models differ from discriminative models?
- **Ans:** Role of Probabilistic Modeling: Probabilistic modeling forms the foundation of generative models, enabling them to learn and represent the underlying data distribution, typically expressed as ( P(X) ) or the joint distribution ( P(X,Y) ). By modeling these probabilities, generative models can capture uncertainty and data variability and generate new samples that are statistically similar to the observed data.
- **Difference Between Generative and Discriminative Models:**

 - Generative Models: These models learn the joint probability distribution ( P(X,Y) ) (or ( p(x) )) and can generate new data instances. They are well-suited for tasks involving data generation, density estimation, and unsupervised learning.

 - Discriminative Models: These models learn the conditional probability distribution ( P(Y \mid X) ) and focus on predicting outputs from given inputs. They are primarily used for classification and regression tasks.

**Q3.** What is the difference between Autoencoders and Variational Autoencoders (VAEs) in the context of text generation?
- **Ans:** Difference Between Autoencoders and Variational Autoencoders (VAEs):
  - **Autoencoders (AEs):** Autoencoders are deterministic models that encode input text into a fixed latent representation and reconstruct the original input. They do not explicitly model the underlying data distribution, limiting their ability to generate diverse or novel text.

  - **Variational Autoencoders (VAEs):** Variational Autoencoders are probabilisic models that learn a distribution over the latent space by representing inputs as parameters (mean and variance). This enables sampling from the latent space, allowing the generation of diverse and coherent text outputs.

**Q4.** Describe the working of attention mechanisms in Neural Machine Translation (NMT). Why are they critical?
- **Ans:**In Neural Machine Translation, attention mechanisms enable the model to dynamically focus on relevant parts of the source sequence during decoding. At each time step, alignment scores are computed between the decoder's current state and the encoder's hidden states. These scores are normalized to obtain attention weights, which are used to generate a context vector as a weighted sum of encoder representations. This context vector informs the generation of each target word.
- **Importance of Attention Mechanisms:**
  - Mitigate the limitations of fixed-length context representations
  - Improve translation accuracy through effective alignment
  - Capture long-range dependencies within sequences
  - Provide interpretability by highlighting relevant input segments

**Q5.** What ethical considerations must be addressed when using gnerative AI for creative content such as poetry or storytelling?
- **Ans:** **Ethical Considerations Include:**
 - Intellectual Property Rights: Ensuring that generated content does not infringe upon copyrighted works or replicate existing authors' styles without proper authorization.
 - Plagiarism and Originality: Maintaining authenticity by avoiding direct or indirect reproduction of existing content, thereby preserving creative integrity.
 - Bias and Fairness: Preventing the propagation of harmful stereotypes or biased narratives that may arise from training data.
 - Transparency: Clearly disclosing the use of AI in the creation process to maintain trust with audiences.
 - Accountability: Establishing responsibility for the content produced, particularly in cases of misinformation or harmful outputs.
 - Cultural sensitivity: Respecting diverse cultures and avoiding offensive or inappropriate representations.



**Q6.** Use the following small text dataset to train a simple Variational Autoencoder (VAE) for text reconstruction:

["The sky is blue", "The sun is bright", "The grass is green",  
"The night is dark", "The stars are shining"]
1. Preprocess the data (tokenize and pad the sequences).
2. Build a basic VAE model for text reconstruction.
3. Train the model and show how it reconstructs or generates similar sentences.
Include your code, explanation, and sample outputs.


In [1]:
# Import and Data Preprocessing
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

texts = [
    "The sky is blue",
    "The sun is bright",
    "The grass is green",
    "The night is dark",
    "The stars are shining"
]

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
max_len = max(len(seq) for seq in sequences)

# Padding
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')

vocab_size = len(tokenizer.word_index) + 1

embedding_dim = 16
latent_dim = 8

# VAE loss layer
class VAELossLayer(tf.keras.layers.Layer):
    def call(self, inputs):
        y_true, y_pred, z_mean, z_log_var = inputs

        # Reconstruction loss
        recon_loss = tf.keras.losses.sparse_categorical_crossentropy(
            y_true, y_pred, from_logits=True
        )
        recon_loss = tf.reduce_mean(recon_loss)

        # KL divergence loss
        kl_loss = -0.5 * tf.reduce_mean(
            1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
        )

        self.add_loss(recon_loss + kl_loss)
        return y_pred

# Build VAE model
# Encoder
inputs = tf.keras.Input(shape=(max_len,))
x = tf.keras.layers.Embedding(vocab_size, embedding_dim)(inputs)
x = tf.keras.layers.Flatten()(x)
h = tf.keras.layers.Dense(32, activation='relu')(x)

z_mean = tf.keras.layers.Dense(latent_dim)(h)
z_log_var = tf.keras.layers.Dense(latent_dim)(h)

# Sampling function
def sampling(args):
    z_mean, z_log_var = args
    epsilon = tf.random.normal(shape=(tf.shape(z_mean)[0], latent_dim))
    return z_mean + tf.exp(0.5 * z_log_var) * epsilon

z = tf.keras.layers.Lambda(sampling)([z_mean, z_log_var])

# Decoder
h_decoded = tf.keras.layers.Dense(32, activation='relu')(z)
outputs = tf.keras.layers.Dense(max_len * vocab_size)(h_decoded)
outputs = tf.keras.layers.Reshape((max_len, vocab_size))(outputs)

# Add VAE Loss
outputs = VAELossLayer()([inputs, outputs, z_mean, z_log_var])

# Model
vae = tf.keras.Model(inputs, outputs)
vae.compile(optimizer='adam')

# Train the model
vae.fit(padded_sequences, padded_sequences,
        epochs=5,
        batch_size=2,
        verbose=1)

# Reconstruction
preds = vae.predict(padded_sequences)
preds = np.argmax(preds, axis=-1)

# Reverse dictionary
index_word = {v: k for k, v in tokenizer.word_index.items()}

def decode(seq):
    return " ".join([index_word.get(i, "") for i in seq if i != 0])

Epoch 1/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - loss: 2.7845
Epoch 2/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 47ms/step - loss: 2.7354
Epoch 3/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - loss: 2.7216 
Epoch 4/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 2.5711
Epoch 5/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 2.5528 
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step


In [2]:
# Output
for i in range(len(preds)):
    print("Original:", texts[i])
    print("Reconstructed:", decode(preds[i]))
    print()

Original: The sky is blue
Reconstructed: sun shining stars dark

Original: The sun is bright
Reconstructed: shining the blue dark

Original: The grass is green
Reconstructed: dark the dark

Original: The night is dark
Reconstructed: sun the blue dark

Original: The stars are shining
Reconstructed: the bright blue dark



**Q7.** Use a pre-trained GPT model (like GPT-2 or GPT-3) to translate a short English paragraph into French and German. Provide the original and translated text.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# Input text
text = "Artificial Intelligence is transforming the world by enabling machines to learn from data and make intelligent decisions."

# Load tokenizer and model for English to French
tokenizer_en_fr = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-fr")
model_en_fr = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-fr")

# Prepare input for French translation
input_ids_fr = tokenizer_en_fr(text, return_tensors="pt").input_ids

# Generate French translation
output_tokens_fr = model_en_fr.generate(input_ids_fr, max_new_tokens=50)
french_translation = tokenizer_en_fr.decode(output_tokens_fr[0], skip_special_tokens=True)

# Load tokenizer and model for English to German
tokenizer_en_de = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")
model_en_de = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-de")

# Prepare input for German translation
input_ids_de = tokenizer_en_de(text, return_tensors="pt").input_ids

# Generate German translation
output_tokens_de = model_en_de.generate(input_ids_de, max_new_tokens=50)
german_translation = tokenizer_en_de.decode(output_tokens_de[0], skip_special_tokens=True)

In [4]:
# Print results
print("Original (English):")
print(text)

print("\nFrench Translation:")
print(french_translation)

print("\nGerman Translation:")
print(german_translation)

Original (English):
Artificial Intelligence is transforming the world by enabling machines to learn from data and make intelligent decisions.

French Translation:
L'intelligence artificielle transforme le monde en permettant aux machines d'apprendre des données et de prendre des décisions intelligentes.

German Translation:
Künstliche Intelligenz transformiert die Welt, indem sie Maschinen ermöglicht, aus Daten zu lernen und intelligente Entscheidungen zu treffen.


**Q8.** Implement a simple attention-based encoder-decoder model for English-to-Spanish translation using Tensorflow or PyTorch.

In [5]:
# Import Libraries
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

eng_sentences = [
    "hello",
    "how are you",
    "i am fine",
    "thank you",
    "good morning"
]

spa_sentences = [
    "<start> hola <end>",
    "<start> como estas <end>",
    "<start> estoy bien <end>",
    "<start> gracias <end>",
    "<start> buenos dias <end>"
]
# Tokenization & Padding
eng_tokenizer = Tokenizer()
spa_tokenizer = Tokenizer(filters='')

eng_tokenizer.fit_on_texts(eng_sentences)
spa_tokenizer.fit_on_texts(spa_sentences)

eng_seq = eng_tokenizer.texts_to_sequences(eng_sentences)
spa_seq = spa_tokenizer.texts_to_sequences(spa_sentences)

max_eng_len = max(len(s) for s in eng_seq)
max_spa_len = max(len(s) for s in spa_seq)

eng_seq = pad_sequences(eng_seq, maxlen=max_eng_len, padding='post')
spa_seq = pad_sequences(spa_seq, maxlen=max_spa_len, padding='post')

eng_vocab = len(eng_tokenizer.word_index) + 1
spa_vocab = len(spa_tokenizer.word_index) + 1

# Encoder
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, units):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(units, return_sequences=True, return_state=True)

    def call(self, x):
        x = self.embedding(x)
        output, state_h, state_c = self.lstm(x)
        return output, state_h, state_c

# Attention Layer
class Attention(tf.keras.layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, encoder_output, hidden):
        hidden = tf.expand_dims(hidden, 1)

        score = self.V(tf.nn.tanh(self.W1(encoder_output) + self.W2(hidden)))
        attention_weights = tf.nn.softmax(score, axis=1)

        context_vector = attention_weights * encoder_output
        context_vector = tf.reduce_sum(context_vector, axis=1)

        return context_vector, attention_weights

# Decoder
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, units):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(units, return_sequences=True, return_state=True)
        self.fc = tf.keras.layers.Dense(vocab_size)
        self.attention = Attention(units)

    def call(self, x, hidden, cell, encoder_output):
        context_vector, attn_weights = self.attention(encoder_output, hidden)

        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)

        output, state_h, state_c = self.lstm(x)
        output = tf.reshape(output, (-1, output.shape[2]))

        x = self.fc(output)

        return x, state_h, state_c

# Intialize the Model
embedding_dim = 32
units = 64

encoder = Encoder(eng_vocab, embedding_dim, units)
decoder = Decoder(spa_vocab, embedding_dim, units)

optimizer = tf.keras.optimizers.Adam()
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# Training Step
@tf.function
def train_step(inp, targ):
    loss = 0

    with tf.GradientTape() as tape:
        enc_out, enc_h, enc_c = encoder(inp)

        dec_h, dec_c = enc_h, enc_c
        start_token = spa_tokenizer.word_index['<start>']
        dec_input = tf.expand_dims([start_token]*inp.shape[0], 1)

        for t in range(1, targ.shape[1]):
            predictions, dec_h, dec_c = decoder(dec_input, dec_h, dec_c, enc_out)
            loss += loss_fn(targ[:, t], predictions)
            dec_input = tf.expand_dims(targ[:, t], 1)

    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))

    return loss

# Training Loop
EPOCHS = 200

for epoch in range(EPOCHS):
    loss = train_step(eng_seq, spa_seq)
    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss {loss.numpy():.4f}")

# Prediction
index_to_word_spa = {v:k for k,v in spa_tokenizer.word_index.items()}

def translate(sentence):
    seq = eng_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_eng_len, padding='post')

    enc_out, enc_h, enc_c = encoder(seq)

    dec_h, dec_c = enc_h, enc_c
    start_token = spa_tokenizer.word_index['<start>']
    end_token = spa_tokenizer.word_index['<end>']
    dec_input = tf.expand_dims([start_token], 1)

    result = []

    for _ in range(max_spa_len):
        predictions, dec_h, dec_c = decoder(dec_input, dec_h, dec_c, enc_out)
        predicted_id = tf.argmax(predictions[0]).numpy()

        if predicted_id == end_token:
            break

        word = index_to_word_spa.get(predicted_id, "")
        if word != '<start>':
             result.append(word)
        dec_input = tf.expand_dims([predicted_id], 1)

    return " ".join(result)

# Sample
print("hello →", translate("hello"))
print("thank you →", translate("thank you"))

Epoch 0, Loss 7.1917
Epoch 50, Loss 5.4244
Epoch 100, Loss 1.2261
Epoch 150, Loss 0.1252
hello → hola
thank you → gracias


**Q9.** Use the following short poetry dataset to simulate poem generation with a pre-trained GPT model:

["Roses are red, violets are blue,",
"Sugar is sweet, and so are you.",
"The moon glows bright in silent skies,",
"A bird sings where the soft wind sighs."]

Using this dataset as a reference for poetic structure and language, generate a new 2-4 line poem using a pre-trained GPT model (such as GPT-2). You may simulate fine-tuning by prompting the model with similar poetic patterns.
Include your code, the prompt used, and the generated poem in your answer.

In [ ]:
from transformers import pipeline

# Load GPT-2 model
generator = pipeline("text-generation", model="gpt2")

# Prompt (simulating fine-tuning using dataset style)
prompt = """Roses are red, violets are blue,
Sugar is sweet, and so are you.
The moon glows bright in silent skies,
A bird sings where the soft wind sighs.

A new poem of 2 to 4 lines, only the poem:"""

# Generate poem
output = generator(
    prompt,
    max_new_tokens=40,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    pad_token_id=generator.tokenizer.eos_token_id
)

In [7]:
# Print result
generated_text = output[0]['generated_text']
start_of_poem_index = generated_text.find(prompt)
if start_of_poem_index != -1:
    generated_poem = generated_text[start_of_poem_index + len(prompt):].strip()
    # Split the generated poem into lines and take only the first 4 lines if more are generated
    poem_lines = generated_poem.split('\n')
     # Filter out empty lines, then take a maximum of 4 lines
    filtered_lines = [line for line in poem_lines if line.strip() != '']
    final_poem = '\n'.join(filtered_lines[:4])

    print("Generated Poem:")
    print(final_poem)
else:
    print("Generated Text (could not separate prompt from poem):")
    print(generated_text)


Generated Poem:
My little girl:
Why, dear,
I can't take that away.
And I can't take that away.



**Q10.** Imagine you are building a creative writing assistant for a publishing company. The assistant should generate story plots and character descriptions usng Generative AI. Describe how you would design the system, including model selection, training data, bias mitigation, and evaluation methods. Explain the real-world challenges you might face.

- **Ans:** **Design of a Creative Writing Assistant Using Generative AI**

A creative writing assistant for a publishing company can be designed as a modular system that leverages advanced generative language models to produce story plots and character descriptions.

**1. Model Selection:**
The system would utilize transformer-based large language models (LLMs) such as GPT or LLaMA due to their strong capabilities in natural language understanding and generation. Fine-tuning or prompt engineering can be applied to adapt the model to specific genres, tones, and publishing standards.

**2. Training Data:**
High-quality datasets consisting of novels, short stories, scripts, and literary content across multiple genres would be used. Data preprocessing steps such as cleaning, deduplication, and annotation (e.g., genre, theme, character roles) are essential to ensure relevance and consistency.

**3. Bias Mitigation:**
Bias can be reduced by using diverse and representative datasets, applying fairness-aware training techniques, and incorporating human review. Content filtering mechanisms should be implemented to avoid harmful, offensive, or stereotypical outputs.

**4. Evaluation Methods:**
Evaluation should combine both automated and human approaches. Automated metrics (e.g., coherence, perplexity) can assess linguistic quality, while human evaluation (editors, writers) ensures creativity, originality, and narrative coherence. A/B testing can also be used to compare outputs.

**5. System Architecture:**
The system may include modules for prompt processing, content generation, post-processing (grammar/style correction), and user feedback integration. A feedback loop can continuously improve model performance.

**6. Real-World Challenges:**
Key challenges include maintaining originality (avoiding plagiarism), controlling hallucinations, ensuring consistent character and plot development, handling ethical concerns, and aligning outputs with editorial standards. Additionally, computational cost, data privacy, and user trust are important considerations.